# <center>Adversarial Validation: Trust Your CV When Train ≠ Test</center>

<center>

![Python](https://img.shields.io/badge/Python-3.10-blue?logo=python&logoColor=white)
![scikit-learn](https://img.shields.io/badge/scikit--learn-1.3-orange?logo=scikit-learn)
![NumPy](https://img.shields.io/badge/NumPy-1.26-013243?logo=numpy)
![pandas](https://img.shields.io/badge/pandas-2.x-150458?logo=pandas)
![License](https://img.shields.io/badge/License-MIT-red)

</center>

---

**Author:** Lorenzo Scaturchio  
**Last Updated:** July 2026  
**Kernel Version:** 1.0

---

## TL;DR

Cross-validation assumes your training and test data come from the same
distribution. On Kaggle they often do **not** — and when they differ, a great
local CV can still miss the leaderboard. **Adversarial validation** is a
two-line idea that catches this: train a classifier to tell *train* rows from
*test* rows. If it can (AUC well above 0.5), your distributions have drifted,
and the same model tells you **which features drifted** and **which training
rows most resemble the test set** — so you can build a validation split that
actually reflects the leaderboard.

This notebook builds a deliberately shifted train/test pair and demonstrates,
with live numbers, the full workflow: detect the shift, localise it, and use it
to construct a trustworthy validation set.

## Table of Contents

1. [Objective](#1.-Objective)
2. [The Idea in One Paragraph](#2.-The-Idea-in-One-Paragraph)
3. [A Deliberately Shifted Dataset](#3.-A-Deliberately-Shifted-Dataset)
4. [The Core Test: Can a Model Separate Train from Test?](#4.-The-Core-Test)
5. [Control: What "No Shift" Looks Like](#5.-Control)
6. [Which Features Drifted?](#6.-Which-Features-Drifted?)
7. [Using It: Build a Test-Like Validation Set](#7.-Using-It)
8. [What To Do When You Detect Shift](#8.-What-To-Do-When-You-Detect-Shift)
9. [Conclusion & Checklist](#9.-Conclusion)

## 1. Objective

"My CV was 0.90 and I dropped 200 places on the leaderboard." Sometimes that is
leakage (see the companion notebook *5 Ways Your Cross-Validation Lies to You*).
Just as often it is **distribution shift**: the test set simply does not look
like the training set, so a validation split carved out of training data is
measuring the wrong thing.

By the end of this notebook you will be able to:

- run **adversarial validation** to get a single number for how far train and
  test have drifted apart;
- read feature importances to find **exactly which columns** shifted;
- select the training rows that most resemble the test set and use them as a
  validation fold that tracks the leaderboard;
- choose a remedy — drop, adapt, or reweight — once you know shift exists.

We use synthetic data with a **known, injected shift**, so we can check that the
method recovers the truth we planted.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import Ridge
from sklearn.model_selection import cross_val_score
from sklearn.metrics import mean_squared_error

SEED = 0
np.random.seed(SEED)

def rmse(y_true, y_pred):
    # version-portable RMSE (older/newer sklearn differ on squared=/root_ helpers)
    return float(np.sqrt(mean_squared_error(y_true, y_pred)))

print("environment ready")

## 2. The Idea in One Paragraph

Stack your training and test feature rows together and throw away the real
target. Create a new label: **0 for "came from train," 1 for "came from test."**
Now train a classifier to predict that label. If the two sets are drawn from
the same distribution, no model can do better than chance (AUC 0.5). If a model
*can* separate them (AUC 0.7, 0.9, ...), the sets differ — and the classifier
has just learned a map of *how* they differ, which its feature importances and
per-row probabilities hand back to you. That is the entire technique.

## 3. A Deliberately Shifted Dataset

Train and test each have 6,000 rows and 8 features. They are identical except
that in **test**, feature `f0` is mean-shifted and `f1` is scaled up. Everything
else matches. A perfect detector should therefore point at `f0` and `f1`.

In [ ]:
n, p = 6000, 8
cols = [f"f{i}" for i in range(p)]

def make(n, shift):
    X = np.random.randn(n, p)
    X[:, 0] += shift            # mean shift on f0
    X[:, 1] *= (1 + 0.6 * shift)  # variance shift on f1
    return X

train = pd.DataFrame(make(n, 0.0), columns=cols)
test  = pd.DataFrame(make(n, 1.0), columns=cols)

print(f"train: {train.shape} | test: {test.shape}")
print("\nfeature means (train vs test):")
print(pd.DataFrame({"train": train.mean(), "test": test.mean()}).round(2).T)

The means table already whispers the answer for `f0`, but real datasets have
dozens of features and shifts hide in variance, correlations, and interactions
that a means table cannot show. Look at the two shifted features directly:

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 3.5), sharey=True)
for ax, feat in zip(axes, ["f0", "f1"]):
    ax.hist(train[feat], bins=60, alpha=0.55, color="#2E7CD6", label="train", density=True)
    ax.hist(test[feat],  bins=60, alpha=0.55, color="#D64550", label="test",  density=True)
    ax.set_title(f"{feat}: train vs test", loc="left")
    ax.spines[["top", "right"]].set_visible(False)
axes[0].legend(frameon=False)
plt.tight_layout(); plt.show()

Two different kinds of drift, side by side: `f0` keeps its shape but slides
right (mean shift), while `f1` keeps its centre and fattens its tails (variance
shift). The second observation is the important one — variance drift barely
registers in a means table, which is why summary statistics alone make an
unreliable detector and we want a classifier that sees the whole joint
distribution. Adversarial validation finds both kinds at once.

## 4. The Core Test: Can a Model Separate Train from Test?

Label every train row 0 and every test row 1, stack them, and cross-validate a
classifier. The AUC *is* the drift score.

In [ ]:
X_all = pd.concat([train, test], ignore_index=True)
is_test = np.r_[np.zeros(len(train)), np.ones(len(test))]

adv_model = RandomForestClassifier(n_estimators=200, random_state=SEED, n_jobs=-1)
adv_auc = cross_val_score(adv_model, X_all, is_test, cv=5, scoring="roc_auc").mean()

print(f"adversarial AUC = {adv_auc:.3f}")
print("0.5 = train and test are indistinguishable")
print("1.0 = perfectly separable (severe shift)")

An AUC around **0.78** — the classifier separates train from test far better
than chance, confirming the distributions differ. As a rough field guide:
**~0.5** is ideal (safe to trust random CV), **0.6-0.8** means meaningful shift
worth handling, and **>0.9** means train and test are nearly disjoint — random
CV will badly mislead you.

Since we control the injected shift, we can also map how the score responds to
severity — useful calibration for reading the number on real data:

In [ ]:
for s in [0.0, 0.25, 0.5, 1.0, 2.0]:
    test_s = pd.DataFrame(make(n, s), columns=cols)
    X_s = pd.concat([train, test_s], ignore_index=True)
    lbl = np.r_[np.zeros(len(train)), np.ones(len(test_s))]
    auc_s = cross_val_score(
        RandomForestClassifier(n_estimators=100, random_state=SEED, n_jobs=-1),
        X_s, lbl, cv=3, scoring="roc_auc").mean()
    print(f"injected shift = {s:4.2f}  ->  adversarial AUC = {auc_s:.3f}")

The response is steep and monotonic: even a quarter-strength shift lifts the
AUC clearly off 0.5, and by shift 2.0 the sets are almost separable. Two
practical consequences. First, the test is *sensitive* — an elevated score is
worth acting on long before it approaches 1.0. Second, the AUC ranks severity
but is not a percentage of drifted rows; use it to compare datasets, features,
or preprocessing variants against each other, not as an absolute quantity.

## 5. Control: What "No Shift" Looks Like

To prove the AUC is measuring drift and not some artefact, re-run against a test
set drawn from the **same** distribution as train (shift = 0).

In [ ]:
test_noshift = pd.DataFrame(make(n, 0.0), columns=cols)
X_ctrl = pd.concat([train, test_noshift], ignore_index=True)
is_test_ctrl = np.r_[np.zeros(len(train)), np.ones(len(test_noshift))]

ctrl_auc = cross_val_score(
    RandomForestClassifier(n_estimators=200, random_state=SEED, n_jobs=-1),
    X_ctrl, is_test_ctrl, cv=5, scoring="roc_auc").mean()

print(f"adversarial AUC, no shift = {ctrl_auc:.3f}   (expected ~0.50)")

~0.50, exactly as it should be: when the distributions match, the best a model
can do is guess. This is also the sanity check to run on your own competition
data first — if even the no-shift case comes back high, you have a leakage or
row-ordering artefact to fix before reading anything into the score.

## 6. Which Features Drifted?

Fit the adversarial model on all the data and read its feature importances. The
features it leans on to tell train from test are precisely the ones that
drifted.

In [ ]:
adv_model.fit(X_all, is_test)
importance = (pd.Series(adv_model.feature_importances_, index=cols)
                .sort_values(ascending=False))

fig, ax = plt.subplots(figsize=(8, 4))
colors = ["#D64550" if f in ("f0", "f1") else "#9AA7B5" for f in importance.index]
bars = ax.barh(importance.index[::-1], importance.values[::-1],
               color=colors[::-1], zorder=3)
ax.set_xlabel("Adversarial-model feature importance")
ax.set_title("Which features let the model tell train from test?", loc="left")
ax.spines[["top", "right"]].set_visible(False)
ax.grid(axis="x", color="#DDDDDD", linewidth=0.6, zorder=0)
plt.tight_layout(); plt.show()

print("top 3 drifting features:")
print(importance.head(3).round(3).to_string())

`f0` and `f1` top the chart (highlighted) — the exact columns we shifted — well
clear of the untouched features (a stray noise feature may drift up the ranking
by chance, but at a fraction of the real signal). On a real dataset this list is
your
to-do: each high-importance feature is either genuinely predictive drift you
must handle, or an artefact (an ID, a timestamp, a row-order proxy) that should
not have been a feature at all.

Importances from one multivariate model can be shared out oddly between
correlated features, so it is worth cross-checking with the simplest possible
view — an adversarial AUC per feature, one column at a time:

In [ ]:
single = {}
for f in cols:
    single[f] = cross_val_score(
        RandomForestClassifier(n_estimators=60, random_state=SEED, n_jobs=-1),
        X_all[[f]], is_test, cv=3, scoring="roc_auc").mean()

single = pd.Series(single).sort_values(ascending=False)
print("single-feature adversarial AUC (0.5 = no drift):")
print(single.round(3).to_string())

The single-feature view agrees with the importances and adds a nuance: `f0`
separates strongly on its own, while `f1` scores lower because a variance
change moves the tails more than the bulk, so univariate separability
understates it. The disagreement pattern is the diagnostic to remember — if
every per-feature AUC sits near 0.5 while the full-model AUC is high, the drift
lives in feature *interactions*, and no single-column fix will remove it.

## 7. Using It: Build a Test-Like Validation Set

The payoff. The adversarial model gives every **training** row a probability of
"looking like test." Rank by it, and the highest-scoring training rows are your
most test-like — a validation fold built from them estimates leaderboard error
far better than a random split does.

We attach a target that depends on the shifted features (so the choice of
validation rows genuinely matters), then compare a **random** holdout against an
**adversarial** holdout by how well each predicts the true error on the real
test set. First, the raw material — the distribution of test-likeness across
training rows:

In [ ]:
# P(test) for each training row, from the adversarial model
p_test = adv_model.predict_proba(train[cols])[:, 1]

cutoff = np.quantile(p_test, 0.75)
fig, ax = plt.subplots(figsize=(8, 3.2))
ax.hist(p_test, bins=50, color="#2E7CD6", zorder=3)
ax.axvline(cutoff, color="#D64550", linestyle="--",
           label=f"top-25% cutoff ({cutoff:.2f}) -> validation candidates")
ax.set_xlabel("P(row looks like test)"); ax.set_ylabel("training rows")
ax.set_title("Test-likeness of the training rows", loc="left")
ax.legend(frameon=False)
ax.spines[["top", "right"]].set_visible(False)
ax.grid(axis="y", color="#DDDDDD", linewidth=0.6, zorder=0)
plt.tight_layout(); plt.show()

print(f"P(test) quartiles: {np.quantile(p_test, [0.25, 0.5, 0.75]).round(3)}")

A broad, single-mode spread: most training rows are somewhat test-like and a
tail is very test-like — which is what a moderate mean/variance shift should
produce. If this histogram were bimodal, with one spike near 0 and one near 1,
that would say the test set contains a sub-population training barely covers,
and no reshuffling of training rows could fully fix validation. The shape is
worth a look before trusting any downstream remedy. Now the payoff experiment:

In [ ]:
# a target that leans on the drifted features
def target(df, rng_seed):
    noise = np.random.RandomState(rng_seed).randn(len(df)) * 0.5
    return df["f0"] * 1.5 + df["f1"] * 0.8 + noise

y_train = target(train, 1)
y_test  = target(test, 2)

# ADVERSARIAL holdout: the 1500 most test-like training rows
order = np.argsort(-p_test)
val_adv, tr_adv = order[:1500], order[1500:]
m_adv = Ridge().fit(train.iloc[tr_adv][cols], y_train.iloc[tr_adv])
adv_val_rmse = rmse(y_train.iloc[val_adv], m_adv.predict(train.iloc[val_adv][cols]))

# RANDOM holdout: 1500 rows at random
perm = np.random.RandomState(3).permutation(len(train))
val_rnd, tr_rnd = perm[:1500], perm[1500:]
m_rnd = Ridge().fit(train.iloc[tr_rnd][cols], y_train.iloc[tr_rnd])
rnd_val_rmse = rmse(y_train.iloc[val_rnd], m_rnd.predict(train.iloc[val_rnd][cols]))

# the ground truth we are trying to estimate
true_test_rmse = rmse(y_test, m_adv.predict(test[cols]))

print(f"mean P(test) of chosen rows: adversarial={p_test[val_adv].mean():.3f}  "
      f"all train={p_test.mean():.3f}")
print()
print(f"random holdout      -> estimated RMSE = {rnd_val_rmse:.3f}")
print(f"adversarial holdout -> estimated RMSE = {adv_val_rmse:.3f}")
print(f"TRUE test RMSE                        = {true_test_rmse:.3f}")
print()
print(f"random-holdout error vs truth      = {rnd_val_rmse - true_test_rmse:+.3f}  (optimistic)")
print(f"adversarial-holdout error vs truth = {adv_val_rmse - true_test_rmse:+.3f}  (on target)")

The adversarial holdout's estimated RMSE lands essentially **on** the true test
RMSE, while the random holdout comes in optimistic — it flatters the model by
validating on rows that do not resemble the test set. The absolute gap here is
small because our injected shift is mild; on real competitions with strong shift
the random holdout can be optimistic by a wide margin, which is exactly the trap
that produces a great CV and a bad leaderboard.

## 8. What To Do When You Detect Shift

A high adversarial AUC is a diagnosis, not a verdict. The remedies, roughly in
order of how often they help:

- **Validate the way we just did** — carve your holdout from the most test-like
  training rows, so local scores track the leaderboard.
- **Drop the offending feature** when a top-importance column is an artefact (an
  ID, an index, a timestamp that just encodes collection order). Removing it
  often *raises* the leaderboard even as it lowers a leaky CV.
- **Reweight training rows** by their P(test) (importance weighting), so the
  model pays more attention to training examples that look like the test set.
- **Adapt the feature** rather than drop it — bin it, rank-transform it, or take
  differences so the shifted raw scale stops mattering.
- **Match your CV scheme to the shift's cause** — if test is a later time
  period, use time-based splits; if it is new entities, use grouped splits.

What you should *not* do is tune hyperparameters against a random-CV score you
now know is measuring the wrong distribution.

The reweighting remedy is two lines, so let us run it rather than describe it:

In [ ]:
# Remedy in action: weight each training row by its odds of being test-like
w = p_test / np.clip(1 - p_test, 1e-3, None)   # importance weights P(test)/P(train)
w = np.clip(w / w.mean(), 0.1, 10.0)           # normalise, clip extreme weights

m_plain    = Ridge().fit(train[cols], y_train)
m_weighted = Ridge().fit(train[cols], y_train, sample_weight=w)

print(f"weight range after clipping: {w.min():.2f} - {w.max():.2f}")
print(f"test RMSE, unweighted training        = {rmse(y_test, m_plain.predict(test[cols])):.3f}")
print(f"test RMSE, importance-weighted        = {rmse(y_test, m_weighted.predict(test[cols])):.3f}")

The two scores land close together — an honest outcome worth understanding. The
relationship we planted is linear and identical in both distributions, so a
correctly-specified model gains nothing from reweighting; the remedy matters
when the model is flexible enough to fit region-specific structure and the
test-like region is under-represented in training. Note the clipping step:
importance weighting carries a bias-variance trade-off, and without the clip a
handful of enormous-weight rows can dominate the fit. One further caveat before
reaching for it on a real competition — if the adversarial AUC is very high
(>0.9), the weights concentrate on a sliver of rows and effectively shrink your
training set. Fixing the validation split, as in Section 7, is usually the
better first move.

Finally, the whole diagnostic packaged as one function you can paste into any
competition pipeline:

In [ ]:
def adversarial_validation(train_df, test_df, features=None, n_estimators=200, seed=SEED):
    """Return (drift AUC, per-feature importances, P(test) per training row)."""
    feats = list(features) if features is not None else list(train_df.columns)
    X = pd.concat([train_df[feats], test_df[feats]], ignore_index=True)
    lbl = np.r_[np.zeros(len(train_df)), np.ones(len(test_df))]
    model = RandomForestClassifier(n_estimators=n_estimators, random_state=seed, n_jobs=-1)
    auc = cross_val_score(model, X, lbl, cv=5, scoring="roc_auc").mean()
    model.fit(X, lbl)
    imp = pd.Series(model.feature_importances_, index=feats).sort_values(ascending=False)
    return auc, imp, model.predict_proba(train_df[feats])[:, 1]

auc_chk, imp_chk, p_chk = adversarial_validation(train, test)
print(f"reusable function, sanity check: AUC = {auc_chk:.3f} (matches Section 4)")
print(f"top drifted features: {list(imp_chk.index[:2])}")

## 9. Conclusion

**Takeaways**

1. Adversarial validation is one move — *classify train vs test* — that turns
   the vague worry "maybe my test set is different" into a number, a ranked list
   of drifted features, and a per-row test-likeness score.
2. On our injected shift it recovered the truth: AUC ~0.78 (vs ~0.50 with no
   shift), importances pointing straight at the two shifted features, and a
   validation split whose error estimate matched the real test error.
3. Always run the **no-shift control** on your own data first; a high AUC there
   means an artefact to fix before interpreting anything.
4. Detection is step one; the value is acting on it — validate test-like, drop
   artefacts, or reweight — instead of trusting a random CV that is quietly
   measuring the wrong thing.

**A pre-submission checklist**

- [ ] Adversarial AUC on train vs test — is it near 0.5, or elevated?
- [ ] No-shift control — does it come back ~0.5 (no artefact)?
- [ ] Top drifted features — genuine signal, or leaky IDs/timestamps to drop?
- [ ] Is your validation fold built from test-like rows, or just random?
- [ ] Does the resulting local score finally move with the leaderboard?

**Next steps to try on your own data**

- Run `adversarial_validation()` on your current competition before the next
  submission — ten minutes of compute either buys trust in random CV or tells
  you exactly what to fix. If the AUC is elevated, I recommend re-validating
  with a test-like holdout first; it is the cheapest change that can improve
  the CV-to-leaderboard correlation.
- Re-run the severity ladder on time-sliced halves of your *training* data to
  see whether drift is growing over time (a sign the test period continues a
  trend your CV ignores).
- Swap the RandomForest for a gradient-boosted model and compare the drift
  AUCs — agreement between two model families is strong evidence the shift is
  real and not one model's quirk.

**Related notebooks in this series:**

- 5 Ways Your Cross-Validation Lies to You
- Feature Engineering Cookbook: 50 Techniques
- Optuna Tuning: A Practical Kaggle Guide

---

**If this notebook helped you trust your CV, please upvote!** Questions welcome
in the comments.

*Lorenzo Scaturchio | July 2026*